**Strategy:**
- **CWE** → structured chunking, one document per CWE entry, fixed fields
- **CVE** → semantic chunking, split long narratives by embedding similarity shift
- **Join** → CVE chunks carry `cwe_id` metadata linking back to parent CWE
- **Output** → `cwe_chunks.parquet` + `cve_chunks.parquet`


### Step 1: Dependencies

In [22]:
%%capture
!pip install sentence-transformers pandas pyarrow requests lxml tqdm faiss-gpu


### Step 2: Configurations

In [3]:
import xml.etree.ElementTree as ET
import json, re, time, math, uuid
from pathlib import Path
from typing import Optional

import requests
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer

CWE_XML_PATH   = "/content/chnk_files/cwec_v4.19.1.xml" # CWE dataset from https://cwe.mitre.org/data/downloads.html
OUT_DIR        = Path("/content/chunks")
OUT_DIR.mkdir(exist_ok=True)

EMBED_MODEL    = "nomic-ai/nomic-embed-text-v2-moe"  #Qwen3-Embedding-8B (Alibaba/Qwen), BGE-M3 (BAAI)
SEMANTIC_THRESHOLD = 0.82   # cosine similarity threshold, below this will add a new chunk
MAX_CVE_PER_CWE    = 30     # cap per CWE

# NVD_API_KEY -- requested from NVD DB https://nvd.nist.gov/developers/request-an-api-key
from google.colab import userdata
NVD_API_KEY = userdata.get('NVD_API_KEY')

### Step 3: Load embedding model

In [4]:
model = SentenceTransformer(EMBED_MODEL, trust_remote_code=True)

def embed(texts: list[str]) -> np.ndarray:
    # nomic-embed-text requires a task prefix
    prefixed = [f"search_document: {t}" for t in texts]
    return model.encode(prefixed, normalize_embeddings=True,
                        show_progress_bar=False, batch_size=64)

print("Model loaded:", EMBED_MODEL)

/root/.cache/huggingface/modules/transformers_modules/nomic_hyphen_ai/nomic_hyphen_bert_hyphen_2048/7710840340a098cfb869c4f65e87cf2b1b70caca/modeling_hf_nomic_bert.py:1634: UserWarning: Install Nomic's megablocks fork for better speed: `pip install git+https://github.com/nomic-ai/megablocks.git`
  warnings.warn("Install Nomic's megablocks fork for better speed: " +


Model loaded: nomic-ai/nomic-embed-text-v2-moe


### Step 4: CWE XML parser (structured chunking)

In [5]:
!pip install xmltodict

In [6]:
NS = {
    'cwe': 'http://cwe.mitre.org/cwe-7',
    'xsi': 'http://www.w3.org/2001/XMLSchema-instance'
}


import xmltodict
def parse_cwe_xml(xml_path):
  xml_string = Path(xml_path).read_text()
  data_dict = xmltodict.parse(xml_string)

  weaknesses_data = data_dict['Weakness_Catalog']['Weaknesses']['Weakness']
  if isinstance(weaknesses_data, dict):
      data_weakness = [weaknesses_data]
  else:
      data_weakness = weaknesses_data

  chunks = [] # Initialize chunks list to store all parsed data

  for dw in data_weakness:
    cwe_id = dw.get('@ID', '')
    name = dw.get('@Name', '')
    wtype = dw.get('@Abstraction', '')  # Base / Variant / Class / Compound
    status = dw.get('@Status', '')

    description = dw.get('Description', '') # Use .get() for safety
    extended = dw.get('Extended_Description', '') # Use .get() for safety

    consequences = []
    consequences_raw = dw.get('Common_Consequences', {}).get('Consequence')
    if consequences_raw is None:
        consequences_list = []
    elif isinstance(consequences_raw, list):
        consequences_list = consequences_raw
    elif isinstance(consequences_raw, dict):
        consequences_list = [consequences_raw]
    else: # If it's a string or other unexpected type, treat as empty for sub-elements
        consequences_list = []

    for c in consequences_list:
        # Assuming c is now always a dictionary
        scope_val  = c.get('Scope', '')
        if isinstance(scope_val, list):
            scope = ", ".join(s or '' for s in scope_val)
        else:
            scope = scope_val

        impact_val = c.get('Impact', '')
        if isinstance(impact_val, list):
            impact = ", ".join(s or '' for s in impact_val)
        else:
            impact = impact_val

        if scope or impact:
            consequences.append(f"{scope}: {impact}".strip(": "))

    mitigations = []
    mitigations_raw = dw.get('Potential_Mitigations', {}).get('Mitigation')
    if mitigations_raw is None:
        mitigations_list = []
    elif isinstance(mitigations_raw, list):
        mitigations_list = mitigations_raw
    elif isinstance(mitigations_raw, dict):
        mitigations_list = [mitigations_raw]
    else: # If it's a string or other unexpected type, treat as empty for sub-elements
        mitigations_list = []

    for m in mitigations_list:
        # Assuming m is now always a dictionary
        phase_val = m.get('Phase', '')
        if isinstance(phase_val, list):
            phase = ", ".join(s or '' for s in phase_val)
        else:
            phase = phase_val

        desc_val = m.get('Description', '')
        if isinstance(desc_val, list):
            desc = ", ".join(s or '' for s in desc_val)
        else:
            desc = desc_val

        if desc:
            mitigations.append(f"[{phase}] {desc}" if phase else desc)

    # Handle Detection_Methods
    detection_methods_raw = dw.get('Detection_Methods', {}).get('Detection_Method')
    detection_texts = []
    if detection_methods_raw is None:
        method_list = []
    elif isinstance(detection_methods_raw, list):
        method_list = detection_methods_raw
    elif isinstance(detection_methods_raw, dict):
        method_list = [detection_methods_raw]
    else: # If it's a string or other unexpected type, treat as empty for sub-elements
        method_list = []

    for dm in method_list:
        # Assuming dm is now always a dictionary
        description_content = dm.get('Description', '')
        # Ensure description_content is a string before appending
        if isinstance(description_content, dict):
            description_content = json.dumps(description_content) # Convert dict to string representation
        elif isinstance(description_content, list):
            description_content = "; ".join([str(d) for d in description_content if d]) # Join list items to string

        if description_content:
            detection_texts.append(str(description_content)) # Ensure it's explicitly a string

    detection = " | ".join(detection_texts[:2]) # cap at 2

    parts = [
        f"CWE-{cwe_id}: {name}",
        f"Type: {wtype}",
        ]
    if description:
        parts.append(f"Description: {description}")
    if extended:
        parts.append(f"Extended: {extended}")
    if consequences:
        parts.append("Consequences: " + " | ".join(consequences))
    if mitigations:
        parts.append("Mitigations: " + " | ".join(mitigations[:3]))  # cap at 3
    if detection:
        parts.append("Detection: " + detection)
    chunk_text = "\n".join(parts)

    chunks.append({
                "chunk_id"     : f"CWE-{cwe_id}",
                "cwe_id"       : f"CWE-{cwe_id}",
                "name"         : name,
                "weakness_type": wtype,
                "status"       : status,
                "chunk_text"   : chunk_text,
                "source"       : "CWE",
            })

  return chunks # Return the list of chunks, not just data_dict


cwe_chunks = parse_cwe_xml(CWE_XML_PATH)


In [7]:
# print(data_test[0])
print(cwe_chunks[0]['chunk_text'][:600])


CWE-1004: Sensitive Cookie Without 'HttpOnly' Flag
Type: Variant
Description: The product uses a cookie to store sensitive information, but the cookie is not marked with the HttpOnly flag.
Extended: The HttpOnly flag directs compatible browsers to prevent client-side script from accessing cookies. Including the HttpOnly flag in the Set-Cookie HTTP response header helps mitigate the risk associated with Cross-Site Scripting (XSS) where an attacker's script code might attempt to read the contents of a cookie and exfiltrate information obtained. When set, browsers that support the flag will not r


### Step 5: NVD API client

In [8]:
NVD_BASE = "https://services.nvd.nist.gov/rest/json/cves/2.0"

def nvd_headers() -> dict:
  h = {}
  if NVD_API_KEY:
    h["apiKey"] = NVD_API_KEY
  return h
results = []
start_index = 0
page_size = 200
params = {
          "cweId"       : "CWE-1004",
          "resultsPerPage": page_size,
          "startIndex"  : start_index,
          }
r = requests.get(NVD_BASE, params=params, headers=nvd_headers(), timeout=30)
print(json.dumps(r.json(), indent=4))

{
    "resultsPerPage": 4,
    "startIndex": 0,
    "totalResults": 4,
    "format": "NVD_CVE",
    "version": "2.0",
    "timestamp": "2026-04-09T16:12:33.780",
    "vulnerabilities": [
        {
            "cve": {
                "id": "CVE-2026-22081",
                "sourceIdentifier": "vdisclose@cert-in.org.in",
                "published": "2026-01-09T12:15:54.260",
                "lastModified": "2026-01-13T14:03:46.203",
                "vulnStatus": "Awaiting Analysis",
                "cveTags": [],
                "descriptions": [
                    {
                        "lang": "en",
                        "value": "This vulnerability exists in Tenda wireless routers (300Mbps Wireless Router F3 and N300 Easy Setup Router) due to the missing HTTPOnly flag for session cookies associated with the web-based administrative interface. A remote at-tacker could exploit this vulnerability by capturing session cookies transmitted over an insecure HTTP connection.\n\nSucces

In [9]:
NVD_BASE = "https://services.nvd.nist.gov/rest/json/cves/2.0"

def nvd_headers() -> dict:
  h = {}
  if NVD_API_KEY: # stored in the Secrets
    h["apiKey"] = NVD_API_KEY
  return h

def fetch_cves_for_cwe(cwe_id: str, max_results: int = MAX_CVE_PER_CWE) -> list[dict]:
  """
  Fetch CVEs linked to a given CWE-ID from NVD API v2.
  Returns a list of raw CVE item dicts.
  """
  results = []
  start_index = 0
  page_size = min(max_results, 20)   # NVD max per page = 2000, we keep small

  while len(results) < max_results:
    params = {
        "cweId"       : cwe_id,
        "resultsPerPage": page_size,
        "startIndex"  : start_index,
    }
    try:
      r = requests.get(NVD_BASE, params=params,
                        headers=nvd_headers(), timeout=30)
      if r.status_code != 200:
        print(f"Request error for {cwe_id}: {r.status_code}")
        break
      data = r.json()
    except Exception as e:
      print(f"Request error for {cwe_id}: {e}")
      break

    vulns = data.get("vulnerabilities", [])
    if not vulns:
      break

    results.extend(vulns)
    total = data.get("totalResults", 0)
    start_index += len(vulns)

    if start_index >= total or len(results) >= max_results:
      break

  return results[:max_results]


def extract_cve_fields(vuln: dict, cwe_id: str) -> Optional[dict]:
  """
  Extract fields from one NVD vulnerability item.
  Returns None if the CVE has no usable description.
  """
  cve = vuln.get("cve", {})
  cve_id = cve.get("id", "")

  descs = cve.get("descriptions", [])
  description = next(
    (d["value"] for d in descs if d.get("lang") == "en"), ""
  )
  if not description or len(description) < 30:
    return None

  # CVSS v4 --> v3.1 --> v3.0 --> v2
  metrics = cve.get("metrics", {})
  cvss_score, cvss_severity, cvss_vector = None, None, None
  for key in ["cvssMetricV40","cvssMetricV31", "cvssMetricV30", "cvssMetricV2"]:
    mlist = metrics.get(key, [])
    if mlist:
      cvss_data = mlist[0].get("cvssData", {})
      cvss_score    = cvss_data.get("baseScore")
      cvss_severity = cvss_data.get("baseSeverity") or mlist[0].get("baseSeverity")
      cvss_vector   = cvss_data.get("vectorString")
      break

  published = cve.get("published", "")[:10]

  return {
          "cve_id"       : cve_id,
          "cwe_id"       : cwe_id,
          "description"  : description,
          "cvss_score"   : cvss_score,
          "cvss_severity": (cvss_severity or "").upper(),
          "cvss_vector"  : cvss_vector or "",
          "published"    : published,
          }


print("NVD API client ready.")

NVD API client ready.


### Step 6: Semantic chunker for CVE narrative text


In [10]:
def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
  return float(np.dot(a, b))

def split_into_sentences(text: str) -> list[str]:
  """Naive but effective sentence splitter for CVE descriptions."""
  # Split on '. ' or '.' followed by uppercase, keep trailing punctuation
  sentences = re.split(r'(?<=[.!?])\s+(?=[A-Z])', text.strip())
  return [s.strip() for s in sentences if len(s.strip()) > 15]

def semantic_chunk_text(
  text: str,
  threshold: float = SEMANTIC_THRESHOLD,
  min_chunk_sentences: int = 2,
) -> list[str]:
  """
  Split a CVE description into semantically coherent chunks.

  Algorithm:
    1. Split into sentences.
    2. Embed each sentence.
    3. Compute cosine similarity between consecutive sentence embeddings.
    4. When similarity drops below `threshold`, start a new chunk.
    5. Merge chunks that are too short (< min_chunk_sentences).

  For most CVE descriptions (1-3 sentences) this returns a single chunk.
  For multi-paragraph advisories it creates meaningful splits.
  """
  sentences = split_into_sentences(text)

  if len(sentences) <= min_chunk_sentences:
    return [text]

  embeddings = embed(sentences)   # (N, D)

  # Build chunk boundaries based on similarity drops
  boundaries = [0]
  for i in range(1, len(sentences)):
    sim = cosine_similarity(embeddings[i-1], embeddings[i])
    if sim < threshold:
      boundaries.append(i)
  boundaries.append(len(sentences))

  # Merge very short chunks into the previous one
  chunks = []
  current = []
  for i, sent in enumerate(sentences):
    current.append(sent)
    is_boundary = (i + 1) in boundaries
    if is_boundary:
      if len(current) < min_chunk_sentences and chunks:
        chunks[-1] = chunks[-1] + " " + " ".join(current)
      else:
        chunks.append(" ".join(current))
      current = []

  return chunks if chunks else [text]

In [11]:
text_clip = """
About half-way between West Egg and New York the motor road hastily joins the railroad and runs beside it for a quarter of a mile, so as to shrink away from a certain desolate area of land. This is a valley of ashes—a fantastic farm where ashes grow like wheat into ridges and hills and grotesque gardens; where ashes take the forms of houses and chimneys and rising smoke and, finally, with a transcendent effort, of men who move dimly and already crumbling through the powdery air. Occasionally a line of gray cars crawls along an invisible track, gives out a ghastly creak, and comes to a rest, and immediately the ash-gray men swarm up with leaden spades and stir up an impenetrable cloud, which screens their obscure operations from your sight.

But above the gray land and the spasms of bleak dust which drift endlessly over it, you perceive, after a moment, the eyes of Doctor T. J. Eckleburg. The eyes of Doctor T. J. Eckleburg are blue and gigantic—their retinas are one yard high. They look out of no face, but, instead, from a pair of enormous yellow spectacles which pass over a non-existent nose. Evidently some wild wag of an ethical ophthalmologist set them there to fatten his practice in the borough of Queens, and then sank down himself into eternal blindness, or forgot them and moved away. But his eyes, dimmed a little by many painless days, under sun and rain, brood on over the solemn dumping ground.

"Terrible place, isn't it," said Tom, exchanging a frown with Doctor Eckleburg.

"Awful."

"It does her good to get away."

"Doesn't her husband object?"

"Wilson? He thinks she goes to see her sister in New York. He’s so dumb he doesn’t know he’s alive."
"""
semantic_chunk_text(text_clip, min_chunk_sentences=1)

['About half-way between West Egg and New York the motor road hastily joins the railroad and runs beside it for a quarter of a mile, so as to shrink away from a certain desolate area of land.',
 'This is a valley of ashes—a fantastic farm where ashes grow like wheat into ridges and hills and grotesque gardens; where ashes take the forms of houses and chimneys and rising smoke and, finally, with a transcendent effort, of men who move dimly and already crumbling through the powdery air.',
 'Occasionally a line of gray cars crawls along an invisible track, gives out a ghastly creak, and comes to a rest, and immediately the ash-gray men swarm up with leaden spades and stir up an impenetrable cloud, which screens their obscure operations from your sight.',
 'But above the gray land and the spasms of bleak dust which drift endlessly over it, you perceive, after a moment, the eyes of Doctor T.',
 'The eyes of Doctor T.',
 'Eckleburg are blue and gigantic—their retinas are one yard high.',
 'T

### Step 7: Combine CWEs and CVEs together with references

In [12]:
parsed_ids = {c['cwe_id'] for c in cwe_chunks}
print(parsed_ids)

{'CWE-1272', 'CWE-1273', 'CWE-1325', 'CWE-174', 'CWE-486', 'CWE-515', 'CWE-538', 'CWE-645', 'CWE-1264', 'CWE-1037', 'CWE-48', 'CWE-15', 'CWE-436', 'CWE-820', 'CWE-761', 'CWE-922', 'CWE-272', 'CWE-597', 'CWE-35', 'CWE-608', 'CWE-581', 'CWE-1268', 'CWE-804', 'CWE-441', 'CWE-475', 'CWE-685', 'CWE-792', 'CWE-799', 'CWE-262', 'CWE-33', 'CWE-293', 'CWE-61', 'CWE-786', 'CWE-1078', 'CWE-172', 'CWE-680', 'CWE-1233', 'CWE-218', 'CWE-334', 'CWE-828', 'CWE-105', 'CWE-1025', 'CWE-1392', 'CWE-1095', 'CWE-176', 'CWE-79', 'CWE-454', 'CWE-1257', 'CWE-1111', 'CWE-1262', 'CWE-37', 'CWE-158', 'CWE-560', 'CWE-555', 'CWE-286', 'CWE-511', 'CWE-795', 'CWE-654', 'CWE-89', 'CWE-433', 'CWE-1281', 'CWE-456', 'CWE-648', 'CWE-1316', 'CWE-1112', 'CWE-311', 'CWE-926', 'CWE-145', 'CWE-544', 'CWE-765', 'CWE-408', 'CWE-1330', 'CWE-703', 'CWE-242', 'CWE-263', 'CWE-589', 'CWE-671', 'CWE-618', 'CWE-59', 'CWE-603', 'CWE-625', 'CWE-447', 'CWE-782', 'CWE-297', 'CWE-667', 'CWE-81', 'CWE-151', 'CWE-1104', 'CWE-1266', 'CWE-705',

In [13]:
USE_TOP_25 = True

# CWE Top 25 Most Dangerous Software Weaknesses (2023 list)
CWE_TOP_25 = [
    "CWE-787", "CWE-79",  "CWE-89",  "CWE-416", "CWE-78",
    "CWE-20",  "CWE-125", "CWE-22",  "CWE-352", "CWE-434",
    "CWE-862", "CWE-476", "CWE-287", "CWE-190", "CWE-502",
    "CWE-77",  "CWE-119", "CWE-798", "CWE-918", "CWE-306",
    "CWE-362", "CWE-269", "CWE-94",  "CWE-863", "CWE-276",
]

parsed_ids = {c['cwe_id'] for c in cwe_chunks}
source_ids = CWE_TOP_25 if USE_TOP_25 else parsed_ids
active_cwe_ids = [cid for cid in source_ids if cid in parsed_ids]

print(f"Fetch CVEs for {len(active_cwe_ids)} CWE IDs:")
print(", ".join(active_cwe_ids))

Fetch CVEs for 25 CWE IDs:
CWE-787, CWE-79, CWE-89, CWE-416, CWE-78, CWE-20, CWE-125, CWE-22, CWE-352, CWE-434, CWE-862, CWE-476, CWE-287, CWE-190, CWE-502, CWE-77, CWE-119, CWE-798, CWE-918, CWE-306, CWE-362, CWE-269, CWE-94, CWE-863, CWE-276


### Step 8: Fetch CVEs from NVD and apply semantic chunking

In [14]:
raw_vulns = fetch_cves_for_cwe("CWE-89")

print(raw_vulns)
print()

for vuln in raw_vulns:
    fields = extract_cve_fields(vuln, "CWE-89")
    print(fields)

# extract_cve_fields(vuln, cwe_id)

[{'cve': {'id': 'CVE-2002-0999', 'sourceIdentifier': 'cve@mitre.org', 'published': '2002-10-04T04:00:00.000', 'lastModified': '2025-04-03T01:03:51.193', 'vulnStatus': 'Deferred', 'cveTags': [], 'descriptions': [{'lang': 'en', 'value': 'Multiple SQL injection vulnerabilities in CARE 2002 before beta 1.0.02 allow remote attackers to perform unauthorized database operations.'}], 'metrics': {'cvssMetricV2': [{'source': 'nvd@nist.gov', 'type': 'Primary', 'cvssData': {'version': '2.0', 'vectorString': 'AV:N/AC:L/Au:N/C:P/I:P/A:P', 'baseScore': 7.5, 'accessVector': 'NETWORK', 'accessComplexity': 'LOW', 'authentication': 'NONE', 'confidentialityImpact': 'PARTIAL', 'integrityImpact': 'PARTIAL', 'availabilityImpact': 'PARTIAL'}, 'baseSeverity': 'HIGH', 'exploitabilityScore': 10.0, 'impactScore': 6.4, 'acInsufInfo': False, 'obtainAllPrivilege': False, 'obtainUserPrivilege': False, 'obtainOtherPrivilege': False, 'userInteractionRequired': False}]}, 'weaknesses': [{'source': 'nvd@nist.gov', 'type':

In [15]:
all_cve_chunks = []
fetch_errors   = []

for cwe_id in tqdm(active_cwe_ids, desc="Fetching CVEs per CWE"):
  raw_vulns = fetch_cves_for_cwe(cwe_id)

  if not raw_vulns:
    fetch_errors.append(cwe_id)
    continue

  for vuln in raw_vulns:
    fields = extract_cve_fields(vuln, cwe_id)
    if fields is None:
      continue

  # Apply semantic chunking to the CVE description
  text_chunks = semantic_chunk_text(fields["description"])

  for chunk_idx, chunk_text in enumerate(text_chunks):
    # Build the full chunk text with contextual header
    severity_tag = ""
    if fields["cvss_score"]:
      severity_tag = f" [CVSS {fields['cvss_score']} {fields['cvss_severity']}]"

    full_text = (
      f"{fields['cve_id']}{severity_tag} — {cwe_id}\n"
      f"{chunk_text}"
    )

    chunk_record = {
        "chunk_id"     : f"{fields['cve_id']}-chunk{chunk_idx}",
        "cve_id"       : fields["cve_id"],
        "cwe_id"       : cwe_id,
        "chunk_index"  : chunk_idx,
        "total_chunks" : len(text_chunks),
        "chunk_text"   : full_text,
        "cvss_score"   : fields["cvss_score"],
        "cvss_severity": fields["cvss_severity"],
        "cvss_vector"  : fields["cvss_vector"],
        "published"    : fields["published"],
        "source"       : "CVE",
    }
    all_cve_chunks.append(chunk_record)

print(f"\nTotal CVE chunks created: {len(all_cve_chunks)}")
if fetch_errors:
  print(f"CWEs with no CVE results: {fetch_errors}")

Fetching CVEs per CWE:   0%|          | 0/25 [00:00<?, ?it/s]


Total CVE chunks created: 25


### Step 9: Validation


In [16]:
active_cwe_chunks = cwe_chunks
print(f"In total {len(active_cwe_chunks)} CWE chunks.")

In total 969 CWE chunks.


In [17]:
df_cwe = pd.DataFrame(active_cwe_chunks)
df_cve = pd.DataFrame(all_cve_chunks)

print("=== CWE chunks ===")
print(df_cwe.shape)
print(df_cwe[["chunk_id", "weakness_type", "status"]].head(5))
print()

print("=== CVE chunks ===")
print(df_cve.shape)
print(df_cve[["chunk_id", "cwe_id", "cvss_severity", "published"]].head(8))
print()

print("=== CVE severity distribution ===")
print(df_cve["cvss_severity"].value_counts())
print()

print("=== CWE status distribution ===")
print(df_cwe["status"].value_counts())

print()
print("=== Sample CWE chunk text ===")
print(df_cwe.iloc[0]["chunk_text"])

print()
print("=== Sample CVE chunk text ===")
print(df_cve.iloc[0]["chunk_text"])

=== CWE chunks ===
(969, 7)
   chunk_id weakness_type      status
0  CWE-1004       Variant  Incomplete
1  CWE-1007          Base  Incomplete
2   CWE-102       Variant  Incomplete
3  CWE-1021          Base  Incomplete
4  CWE-1022       Variant  Incomplete

=== CVE chunks ===
(25, 11)
               chunk_id   cwe_id cvss_severity   published
0  CVE-2009-0269-chunk0  CWE-787        MEDIUM  2009-01-26
1  CVE-2002-2377-chunk0   CWE-79        MEDIUM  2002-12-31
2  CVE-2004-2754-chunk0   CWE-89          HIGH  2004-12-31
3  CVE-2010-1824-chunk0  CWE-416          HIGH  2010-09-24
4  CVE-2008-6669-chunk0   CWE-78          HIGH  2009-04-08
5  CVE-2002-2314-chunk0   CWE-20        MEDIUM  2002-12-31
6  CVE-2011-3906-chunk0  CWE-125        MEDIUM  2011-12-13
7  CVE-2003-1465-chunk0   CWE-22        MEDIUM  2003-12-31

=== CVE severity distribution ===
cvss_severity
HIGH        12
MEDIUM      10
CRITICAL     2
LOW          1
Name: count, dtype: int64

=== CWE status distribution ===
status
Incomplet

### Step 10: Wrap up all chunks

In [30]:
def batch_embed_df(df: pd.DataFrame, text_col: str = "chunk_text",
                   batch_size: int = 64) -> np.ndarray:
  """Embed all rows in a DataFrame column, return (N, D) array."""
  texts = df[text_col].tolist()
  all_embs = []
  for i in tqdm(range(0, len(texts), batch_size), desc="Embedding"):
    batch = texts[i:i+batch_size]
    embs  = embed(batch)
    all_embs.append(embs)
  return np.vstack(all_embs)

print("Embedding CWE chunks...")
cwe_embeddings = batch_embed_df(df_cwe)
print(f"  Shape: {cwe_embeddings.shape}")

print("Embedding CVE chunks...")
cve_embeddings = batch_embed_df(df_cve)
print(f"  Shape: {cve_embeddings.shape}")

Embedding CWE chunks...


Embedding:   0%|          | 0/16 [00:00<?, ?it/s]

  Shape: (969, 768)
Embedding CVE chunks...


Embedding:   0%|          | 0/1 [00:00<?, ?it/s]

  Shape: (25, 768)


In [32]:
"""
Save chunk metadata as Parquet
969 rows of metadata (chunk_id, name, chunk_text, etc.)
"""
cwe_parquet = OUT_DIR / "cwe_chunks.parquet"
cve_parquet = OUT_DIR / "cve_chunks.parquet"
df_cwe.to_parquet(cwe_parquet, index=False)
df_cve.to_parquet(cve_parquet, index=False)

"""
969×768 matrix of float32 vectors.
This is the embedding matrix.
"""
np.save(OUT_DIR / "cwe_embeddings.npy", cwe_embeddings)
np.save(OUT_DIR / "cve_embeddings.npy", cve_embeddings)

# Manifest
manifest = {
    "embed_model"    : EMBED_MODEL,
    "embed_dim"      : int(cwe_embeddings.shape[1]),
    "semantic_threshold": SEMANTIC_THRESHOLD,
    "cwe_chunks"     : len(df_cwe),
    "cve_chunks"     : len(df_cve),
    "cwe_ids_covered": sorted(df_cve["cwe_id"].unique().tolist()),
}
(OUT_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2))

print("Saved to", OUT_DIR)
!ls -lh /content/chunks/

Saved to /content/chunks
total 6.4M
-rw-r--r-- 1 root root  14K Apr  9 18:19 cve_chunks.parquet
-rw-r--r-- 1 root root  76K Apr  9 16:22 cve_embeddings.index
-rw-r--r-- 1 root root  76K Apr  9 18:19 cve_embeddings.npy
-rw-r--r-- 1 root root 537K Apr  9 18:19 cwe_chunks.parquet
-rw-r--r-- 1 root root 2.9M Apr  9 16:22 cwe_embeddings.index
-rw-r--r-- 1 root root 2.9M Apr  9 18:19 cwe_embeddings.npy
-rw-r--r-- 1 root root  541 Apr  9 18:19 manifest.json


Using Faiss

In [25]:
!pip install faiss-gpu-cu12

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 MB 69.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 581.2/581.2 MB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 43.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.6/89.6 MB 17.6 MB/s eta 0:00:00


In [26]:
import faiss

def build_faiss_index(df: pd.DataFrame, text_col: str = "chunk_text",
                      batch_size: int = 64) -> faiss.IndexFlatIP:
    """Embed all rows and build a FAISS IndexFlatIP (cosine on unit vectors)."""
    texts = df[text_col].tolist()
    all_embs = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Embedding"):
        batch = texts[i:i+batch_size]
        embs  = embed(batch)
        all_embs.append(embs)
    matrix = np.vstack(all_embs).astype("float32")
    dim    = matrix.shape[1]
    index  = faiss.IndexFlatIP(dim)
    index.add(matrix)
    return index, matrix

print("Building FAISS index for CWE chunks...")
cwe_index, cwe_embeddings = build_faiss_index(df_cwe)
print(f"  CWE index: {cwe_index.ntotal} vectors, dim {cwe_index.d}")

print("Building FAISS index for CVE chunks...")
cve_index, cve_embeddings = build_faiss_index(df_cve)
print(f"  CVE index: {cve_index.ntotal} vectors, dim {cve_index.d}")


Building FAISS index for CWE chunks...


Embedding:   0%|          | 0/16 [00:00<?, ?it/s]

  CWE index: 969 vectors, dim 768
Building FAISS index for CVE chunks...


Embedding:   0%|          | 0/1 [00:00<?, ?it/s]

  CVE index: 25 vectors, dim 768


In [28]:
cwe_parquet = OUT_DIR / "cwe_chunks.parquet"
cve_parquet = OUT_DIR / "cve_chunks.parquet"
df_cwe.to_parquet(cwe_parquet, index=False)
df_cve.to_parquet(cve_parquet, index=False)

faiss.write_index(cwe_index, str(OUT_DIR / "cwe_embeddings.index"))
faiss.write_index(cve_index, str(OUT_DIR / "cve_embeddings.index"))

manifest = {
    "embed_model"       : EMBED_MODEL,
    "embed_dim"         : cwe_index.d,
    "semantic_threshold": SEMANTIC_THRESHOLD,
    "index_type"        : "IndexFlatIP",
    "cwe_chunks"        : len(df_cwe),
    "cve_chunks"        : len(df_cve),
    "cwe_ids_covered"   : sorted(df_cve["cwe_id"].unique().tolist()),
}
(OUT_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2))

print("Saved to", OUT_DIR)
!ls -lh /content/chunks/

Saved to /content/chunks
total 3.5M
-rw-r--r-- 1 root root  14K Apr  9 16:22 cve_chunks.parquet
-rw-r--r-- 1 root root  76K Apr  9 16:22 cve_embeddings.index
-rw-r--r-- 1 root root 537K Apr  9 16:22 cwe_chunks.parquet
-rw-r--r-- 1 root root 2.9M Apr  9 16:22 cwe_embeddings.index
-rw-r--r-- 1 root root  572 Apr  9 16:22 manifest.json


In [33]:
!zip -r /content/rag_chunks.zip /content/chunks/

from google.colab import files
files.download('/content/rag_chunks.zip')

updating: content/chunks/ (stored 0%)
updating: content/chunks/manifest.json (deflated 62%)
updating: content/chunks/cwe_chunks.parquet (deflated 16%)
updating: content/chunks/cve_embeddings.index (deflated 7%)
updating: content/chunks/cve_chunks.parquet (deflated 46%)
updating: content/chunks/cwe_embeddings.index (deflated 7%)
  adding: content/chunks/cve_embeddings.npy (deflated 7%)
  adding: content/chunks/cwe_embeddings.npy (deflated 7%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>